# Genome-wide screen preprocessing

Optional reconstruction from the large single-cell file, available on request, and its annotation workbook. The steps aggregate median morphology per well, annotate genes, remove visually rejected samples, rescale plate medians, add variability/fraction features, nullify outlying channels, and retain wells with at least 50 cells. Results remain in memory. Downstream figures start from S3 and do not require this step.

In [ ]:
from pathlib import Path
import numpy as np
import numpy.typing
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PACKAGE = Path.cwd()
if not (PACKAGE / "Datasets").is_dir():
    PACKAGE = PACKAGE.parent
DATASETS = PACKAGE / "Datasets"
if not DATASETS.is_dir():
    raise FileNotFoundError("Launch this notebook from the package root or Github folder.")
plt.rcParams.update({"font.family": "Arial", "text.usetex": False})

import re
from typing import List, Tuple, Dict
from pycytominer import aggregate
RUN_PREPROCESSING = False
SINGLE_CELL_INPUT = Path("path/to/single-cell_profiles_keio_stat.parquet")
ANNOTATION_INPUT = Path("path/to/metadata_sorted_annotated.xlsx")
if RUN_PREPROCESSING and not (SINGLE_CELL_INPUT.is_file() and ANNOTATION_INPUT.is_file()):
    raise FileNotFoundError("Set the paths to the on-request single-cell file and its annotation workbook.")

In [ ]:
def add_cog_annotations(agg_df: pd.DataFrame, metadata_annotation: pd.DataFrame) -> pd.DataFrame:
    """
    Add COG annotations to aggregated dataframe.
    
    Parameters:
    -----------
    agg_df : pd.DataFrame
        Aggregated well-level dataframe
    metadata_annotation : pd.DataFrame
        Annotation dataframe with Metadata_gene and Metadata_COG columns
    
    Returns:
    --------
    pd.DataFrame
        Dataframe with COG annotations added
    """
    cog_grouped = metadata_annotation.groupby('Metadata_gene')['Metadata_COG'].apply(list).reset_index()
    agg_ann_df = agg_df.merge(cog_grouped, on='Metadata_gene', how='left')
    return agg_ann_df

def add_cv_to_agg(df, agg_df, features, groupby_cols=['Metadata_plate', 'Metadata_gene', 'Metadata_well']):
    """
    Compute coefficient of variation (std/mean) for selected features from single-cell df
    and merge them into the aggregated agg_df.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Single-cell dataframe with raw features.
    agg_df : pandas.DataFrame
        Aggregated dataframe (e.g. from pycytominer.aggregate).
    features : list of str
        Features for which to compute CV.
    groupby_cols : list of str
        Columns to group by (default: ['Metadata_plate', 'Metadata_gene', 'Metadata_well']).
    
    Returns
    -------
    pandas.DataFrame
        Aggregated dataframe with one CV column per feature, values specific to each group.
    """
    stats = df.groupby(groupby_cols)[features].agg(['mean', 'std'])
    cv_df = pd.DataFrame(index=stats.index)
    for feat in features:
        cv_df[f'{feat}_CV'] = stats[feat, 'std'] / stats[feat, 'mean']
    cv_df = cv_df.reset_index()
    merged = agg_df.merge(cv_df, on=groupby_cols, how='left')
    return merged

def add_robust_cv_to_agg(df, agg_df, features, groupby_cols=('Metadata_plate', 'Metadata_gene', 'Metadata_well'), mad_scale=1.4826):
    """
    Compute outlier-robust coefficient of variation for selected features from
    single-cell df and merge them into agg_df.

    Robust CV is calculated as:

        robust_CV = scaled_MAD / abs(median)

    where:

        scaled_MAD = 1.4826 * median(abs(x - median(x)))

    The 1.4826 factor makes MAD comparable to standard deviation for normally
    distributed data.

    Parameters
    ----------
    df : pandas.DataFrame
        Single-cell dataframe with raw features.
    agg_df : pandas.DataFrame
        Aggregated dataframe.
    features : list of str
        Features for which to compute robust CV.
    groupby_cols : list of str or tuple of str
        Columns to group by.
    mad_scale : float
        Scaling factor for MAD. Default is 1.4826.

    Returns
    -------
    pandas.DataFrame
        Aggregated dataframe with one robust CV column per feature.
    """
    groupby_cols = list(groupby_cols)

    def robust_cv(x):
        x = x.dropna()
        if len(x) == 0:
            return np.nan
        median = x.median()
        if median == 0:
            return np.nan
        mad = np.median(np.abs(x - median))
        robust_std = mad_scale * mad
        return robust_std / abs(median)
    cv_df = df.groupby(groupby_cols)[features].agg(robust_cv).reset_index()
    cv_df = cv_df.rename(columns={feat: f'{feat}_robust_CV' for feat in features})
    merged = agg_df.merge(cv_df, on=groupby_cols, how='left')
    return merged

def compute_feature_fraction(df, feature, threshold, direction='above'):
    """
    Computes the fraction of rows per group where the feature is above or below a threshold.

    Parameters:
    - df (pd.DataFrame): The input DataFrame.
    - feature (str): The feature column to evaluate.
    - threshold (float): The threshold value for comparison.
    - direction (str): 'above' to compute fraction of values above the threshold,
                       'below' to compute fraction of values below the threshold.

    Returns:
    - pd.DataFrame: A DataFrame with columns 'Metadata_plate', 'Metadata_gene', 'Metadata_well',
                    and the fraction column.
    """
    if direction not in ['above', 'below']:
        raise ValueError("Parameter 'direction' must be either 'above' or 'below'.")
    subset_df = df[['Metadata_plate', 'Metadata_gene', 'Metadata_well', feature]].copy()
    if direction == 'above':
        condition = subset_df[feature] > threshold
        fraction_column_name = f'fraction_{feature}_above_{threshold}'
    else:
        condition = subset_df[feature] <= threshold
        fraction_column_name = f'fraction_{feature}_below_{threshold}'
    group_cols = ['Metadata_plate', 'Metadata_gene', 'Metadata_well']
    count_satisfying = subset_df[condition].groupby(group_cols).size()
    total_count = subset_df.groupby(group_cols).size()
    fraction = (count_satisfying / total_count).fillna(0).reset_index(name=fraction_column_name)
    return fraction

def compute_fraction_features_batch(df_single_cell: pd.DataFrame, agg_df: pd.DataFrame, fraction_specs: List[Dict]) -> pd.DataFrame:
    """
    Compute multiple fraction features and merge all at once.
    
    Parameters:
    -----------
    df_single_cell : pd.DataFrame
        Original single-cell dataframe
    agg_df : pd.DataFrame
        Aggregated well-level dataframe
    fraction_specs : List[Dict]
        List of dicts with keys: 'feature', 'threshold', 'direction'
    
    Returns:
    --------
    pd.DataFrame
        Aggregated dataframe with all fraction features added
    """
    result_df = agg_df.copy()
    for spec in fraction_specs:
        fraction_df = compute_feature_fraction(df_single_cell, spec['feature'], spec['threshold'], spec['direction'])
        result_df = result_df.merge(fraction_df, on=['Metadata_plate', 'Metadata_gene', 'Metadata_well'], how='left')
    return result_df

def add_std_feature(df_single_cell: pd.DataFrame, agg_df: pd.DataFrame, feature: str, new_name: str=None) -> pd.DataFrame:
    """
    Add standard deviation of a feature from single-cell data to aggregated dataframe.
    
    Parameters:
    -----------
    df_single_cell : pd.DataFrame
        Original single-cell dataframe
    agg_df : pd.DataFrame
        Aggregated well-level dataframe
    feature : str
        Feature to compute std for
    new_name : str, optional
        Name for the new std column
    
    Returns:
    --------
    pd.DataFrame
        Aggregated dataframe with std feature added
    """
    grouping_cols = ['Metadata_plate', 'Metadata_gene', 'Metadata_well']
    if new_name is None:
        new_name = f'{feature}_std'
    std_df = df_single_cell.groupby(grouping_cols)[feature].std().reset_index()
    std_df = std_df.rename(columns={feature: new_name})
    result_df = agg_df.merge(std_df, on=grouping_cols, how='left')
    return result_df

def clean_feature_outliers(df, features, metadata_gene_col='Metadata_gene', metadata_plate_col='Metadata_plate', metadata_well_col='Metadata_well', threshold=3, apply_nullification=True, nullify_related_C_features=True, df_cleaned=None):
    """
    Cleans outlier WT wells by z-score and optionally nullifies the original and related C# features.

    Parameters:
    - df: Original DataFrame.
    - features: str or list of feature names to clean.
    - metadata_gene_col: Gene label column.
    - metadata_plate_col: Plate column.
    - metadata_well_col: Well column.
    - threshold: Z-score threshold.
    - apply_nullification: If True, set NaNs where WT outliers exist.
    - nullify_related_C_features: If True, nullify all features starting with C# prefix.
    - df_cleaned: Optional existing cleaned DataFrame to continue modifying.

    Returns:
    - df_cleaned: DataFrame with updated NaNs.
    - all_wt_outliers: Dict mapping each feature to its WT outlier DataFrame.
    """
    if isinstance(features, str):
        features = [features]
    if df_cleaned is None:
        df_cleaned = df.copy()
    all_wt_outliers = {}
    for feature in features:
        df_filtered = df[np.isfinite(df[feature])].copy()
        if df_filtered.empty:
            continue
        wt_data = df_filtered[df_filtered[metadata_gene_col] == 'WT'].copy()
        if wt_data.empty:
            continue
        wt_mean = wt_data[feature].mean()
        wt_std = wt_data[feature].std()
        df_filtered[f'z_{feature}'] = np.abs((df_filtered[feature] - wt_mean) / wt_std)
        wt_outliers = df_filtered[(df_filtered[metadata_gene_col] == 'WT') & (df_filtered[f'z_{feature}'] > threshold)]
        all_wt_outliers[feature] = wt_outliers
        existing_nans_before = df_cleaned[feature].isna().sum()
        if apply_nullification:
            for (_, row) in wt_outliers.iterrows():
                plate = row[metadata_plate_col]
                well = row[metadata_well_col]
                n_wts_on_plate = df[(df[metadata_plate_col] == plate) & (df[metadata_gene_col] == 'WT')].shape[0]
                if n_wts_on_plate == 1:
                    df_cleaned.loc[df_cleaned[metadata_plate_col] == plate, feature] = np.nan
                else:
                    df_cleaned.loc[(df_cleaned[metadata_plate_col] == plate) & (df_cleaned[metadata_well_col] == well), feature] = np.nan
            wt_outlier_counts = wt_outliers.groupby(metadata_plate_col).size()
            plates_to_nullify = wt_outlier_counts[wt_outlier_counts >= 2].index
            df_cleaned.loc[df_cleaned[metadata_plate_col].isin(plates_to_nullify), feature] = np.nan
            if nullify_related_C_features:
                match = re.search('(?:fraction_)?C(\\d+)', feature)
                if match:
                    chan_num = match.group(1)
                    patterns = [f'C{chan_num}', f'fraction_C{chan_num}']
                    related_cols = [col for col in df_cleaned.columns if any((col.startswith(p) for p in patterns))]
                    if related_cols:
                        for col in related_cols:
                            df_cleaned.loc[df_cleaned[metadata_plate_col].isin(plates_to_nullify), col] = np.nan
            total_nans_after = df_cleaned[feature].isna().sum()
            new_nans = total_nans_after - existing_nans_before
    return (df_cleaned, all_wt_outliers)

def nullify_channel_features(df, channel, genes):
    """
    Set all features belonging to a given channel (e.g. 'C3')
    to NaN for selected Metadata_gene(s).

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing Metadata_gene and feature columns
    channel : str
        Channel prefix, e.g. 'C3'
    genes : str or list of str
        Metadata_gene(s) for which features should be nullified

    Returns
    -------
    pd.DataFrame
        Copy of df with channel features set to NaN for selected genes
    """
    if isinstance(genes, str):
        genes = [genes]
    df_out = df.copy()
    channel_cols = [c for c in df_out.columns if c.startswith(f'{channel}_')]
    gene_mask = df_out['Metadata_gene'].isin(genes)
    df_out.loc[gene_mask, channel_cols] = np.nan
    return df_out

def add_cell_counts(df_single_cell: pd.DataFrame, agg_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add cell count per well to aggregated dataframe.
    """
    grouping_cols = ['Metadata_plate', 'Metadata_gene', 'Metadata_well']
    cell_counts = df_single_cell.groupby(grouping_cols).size().reset_index(name='Metadata_cell_count')
    result_df = agg_df.merge(cell_counts, on=grouping_cols, how='left')
    result_df['Metadata_cell_count'] = result_df['Metadata_cell_count'].fillna(0).astype(int)
    return result_df

def validate_wt_presence(df: pd.DataFrame, wt_label: str='WT') -> bool:
    """
    Check if all plates contain at least one WT sample.
    """
    wt_by_plate = df.groupby('Metadata_plate')['Metadata_gene'].apply(lambda g: (g == wt_label).any())
    plates_without_wt = wt_by_plate[~wt_by_plate]
    if plates_without_wt.empty:
        return True
    else:
        return False

## Parameters and workflow
CV is sample SD/mean; robust CV is 1.4826 MAD/median. Fraction features use the comparisons in the specifications. Positive plate features are rescaled multiplicatively in log space (epsilon 1e-6), signed features additively. Cleaning thresholds and manually rejected channels follow the source workflow.

In [ ]:
cv_features = ['C4_cell_total_obj_rect_length', 'C4_cell_avg_obj_rect_width', 'C4_cell_total_obj_volume_approx', 'C4_cell_avg_obj_aspectratio', 'C4_cell_avg_obj_circularity', 'C4_cell_avg_obj_SOV', 'C4_cell_avg_obj_mean_c', 'C4_NC_ratio', 'C4_object_number', 'C5_cell_total_obj_rect_length', 'C5_cell_avg_obj_rect_width', 'C5_cell_total_obj_volume_approx', 'C5_cell_avg_obj_aspectratio', 'C5_cell_avg_obj_circularity', 'C5_cell_avg_obj_SOV', 'C5_cell_avg_obj_mean_c', 'C5_NC_ratio', 'C5_object_number', 'C3_NC_ratio', 'C2_NC_ratio']
robust_cv_features = ['cell_length', 'cell_width', 'cell_volume', 'cell_surface_area', 'cell_aspect_ratio', 'cell_circularity', 'cell_SOV', 'cell_mean_contour_curvature']

fraction_specs = [{'feature': 'cell_constriction_degree_width_axial', 'threshold': 0.1, 'direction': 'above'}, {'feature': 'C5_object_number', 'threshold': 1, 'direction': 'above'}, {'feature': 'C3_NC_ratio', 'threshold': 0.6, 'direction': 'below'}, {'feature': 'C2_NC_ratio', 'threshold': 0.8, 'direction': 'below'}]

outlier_cleaning_steps = [{'name': 'Round 1: NC ratios and nucleoid position', 'features': ['C2_NC_ratio', 'C3_NC_ratio', 'C4_NC_ratio', 'C5_std_object_avg_center_distance', 'C2_NC_ratio_CV', 'C3_NC_ratio_CV'], 'threshold': 3.0, 'nullify_related_C_features': True}, {'name': 'Round 3: Fraction features', 'features': ['fraction_cell_constriction_degree_width_axial_above_0.1', 'fraction_C5_object_number_above_1', 'fraction_C3_NC_ratio_below_0.6', 'fraction_C2_NC_ratio_below_0.8'], 'threshold': 3.0, 'nullify_related_C_features': True}, {'name': 'Round 4: Cell size variability', 'features': ['cell_width_robust_CV', 'cell_length_robust_CV'], 'threshold': 3.0, 'nullify_related_C_features': False}, {'name': 'Round 5: Constriction and volume (lenient)', 'features': ['cell_constriction_degree_width_axial', 'cell_volume'], 'threshold': 5.0, 'nullify_related_C_features': False}]

min_cell_count = 50
max_nan_fraction = 0.8

In [ ]:
if RUN_PREPROCESSING:
    df_annotated = pd.read_parquet(SINGLE_CELL_INPUT)
    metadata_annotation = pd.read_excel(ANNOTATION_INPUT)
    df_annotated['_plate_well'] = df_annotated['Metadata_plate'].astype(str) + '_' + df_annotated['Metadata_well'].astype(str)
    metadata_annotation['_plate_well'] = metadata_annotation['Metadata_plate'].astype(str) + '_' + metadata_annotation['Metadata_well'].astype(str)
    gene_map = metadata_annotation.drop_duplicates('_plate_well').set_index('_plate_well')['Metadata_gene']
    df_annotated['Metadata_gene'] = df_annotated['_plate_well'].map(gene_map)
    df_annotated['Metadata_gene'] = df_annotated['Metadata_gene'].astype(str).str.strip().replace({'0': 'WT', '0.0': 'WT', 0: 'WT'})
    df_annotated = df_annotated.drop(columns='_plate_well')
    cols = df_annotated.columns.tolist()
    cols.remove('Metadata_gene')
    well_idx = cols.index('Metadata_well')
    cols = cols[:well_idx + 1] + ['Metadata_gene'] + cols[well_idx + 1:]
    df_annotated = df_annotated[cols]

    metadata_columns = df_annotated.filter(regex='^Metadata_')
    features = [col for col in df_annotated.columns if col not in metadata_columns.columns]
    agg_well_df = aggregate(population_df=df_annotated, strata=['Metadata_plate', 'Metadata_gene', 'Metadata_well'], features=features, operation='median')
    agg_well_df.head()

    agg_ann_well_df = add_cog_annotations(agg_well_df, metadata_annotation)
    agg_ann_well_df.head()

    genes_to_remove = ['yhjJ', 'xdhA', 'yqeC', 'ygeW', 'yijE', 'ygeM', 'yhdY', 'ygfK', 'pgi', 'ypdA', 'yehE', 'ydhT', 'yajR', 'nfuA', 'yigL', 'mgrB', 'dnaT', 'ynfL', 'ydiH', 'yigG', 'efp', 'yphE', 'dadA']
    agg_ann_well_df = agg_ann_well_df.loc[~agg_ann_well_df['Metadata_gene'].isin(genes_to_remove)].copy()

    import numpy as np
    import pandas as pd
    features = [c for c in agg_ann_well_df.columns if not c.startswith('Metadata_') and pd.api.types.is_numeric_dtype(agg_ann_well_df[c])]
    epsilon = 1e-06
    positive_features = [c for c in features if agg_ann_well_df[c].min() >= 0]
    signed_features = [c for c in features if c not in positive_features]
    global_log_median = np.log(agg_ann_well_df[positive_features] + epsilon).median()
    global_median_signed = agg_ann_well_df[signed_features].median()
    rescaled_dfs = []
    for (plate, plate_df) in agg_ann_well_df.groupby('Metadata_plate', sort=False):
        plate_df = plate_df.copy()
        if positive_features:
            plate_log = np.log(plate_df[positive_features] + epsilon)
            plate_log_median = plate_log.median()
            plate_df[positive_features] = np.exp(plate_log.sub(plate_log_median, axis=1).add(global_log_median, axis=1)) - epsilon
        if signed_features:
            plate_median = plate_df[signed_features].median()
            plate_df[signed_features] = plate_df[signed_features].sub(plate_median, axis=1).add(global_median_signed, axis=1)
        rescaled_dfs.append(plate_df)
    rescaled_df = pd.concat(rescaled_dfs, ignore_index=True)

    rescaled_df = add_cv_to_agg(df_annotated, rescaled_df, cv_features)
    cv_cols = [col for col in rescaled_df.columns if col.endswith('_CV')]
    rescaled_df = add_robust_cv_to_agg(df_annotated, rescaled_df, robust_cv_features)
    robust_cv_cols = [col for col in rescaled_df.columns if col.endswith('_robust_CV')]

    rescaled_df = compute_fraction_features_batch(df_annotated, rescaled_df, fraction_specs)
    frac_cols = [col for col in rescaled_df.columns if col.startswith('fraction_')]

    rescaled_df = add_std_feature(df_annotated, rescaled_df, 'C5_object_avg_center_distance', 'C5_std_object_avg_center_distance')

    agg_ann_well_cleaned_df = rescaled_df.copy()
    all_outlier_dfs = []
    for (i, cleaning_step) in enumerate(outlier_cleaning_steps, 1):
        (agg_ann_well_cleaned_df, outliers) = clean_feature_outliers(df=rescaled_df, features=cleaning_step['features'], threshold=cleaning_step['threshold'], apply_nullification=True, nullify_related_C_features=cleaning_step.get('nullify_related_C_features', True), df_cleaned=agg_ann_well_cleaned_df)
        if outliers:
            all_outlier_dfs.append(outliers)

    agg_ann_well_cleaned_df = nullify_channel_features(df=agg_ann_well_cleaned_df, channel='C3', genes=['fepE', 'mmuM', 'caiC', 'copA', 'caiA', 'metH', 'ydhS', 'yedJ'])
    agg_ann_well_cleaned_df = nullify_channel_features(df=agg_ann_well_cleaned_df, channel='C5', genes=['ydeA'])

    agg_ann_well_cleaned_df = add_cell_counts(df_annotated, agg_ann_well_cleaned_df)

    before = len(agg_ann_well_cleaned_df)
    agg_ann_well_cleaned_df = agg_ann_well_cleaned_df[agg_ann_well_cleaned_df['Metadata_cell_count'] >= min_cell_count]
    after = len(agg_ann_well_cleaned_df)

    validate_wt_presence(agg_ann_well_cleaned_df, wt_label='WT')
    display(agg_ann_well_cleaned_df.head())
else:
    print("Raw preprocessing is optional; downstream analyses use Dataset S3.")